In [1]:
# =====================================
# Setup (Colab)
# =====================================
!nvidia-smi   # Check GPU availability

import numpy as np
import time
import cv2
from numba import cuda

print("CUDA available:", cuda.is_available())


Thu Sep 25 16:22:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Update and install dependencies
!apt-get update -qq
!apt-get install -y build-essential pkg-config libopencv-dev

# Install CUDA toolkit 12.4
!apt-get install -y cuda-toolkit-12-4

# Set environment variables for CUDA 12.4
import os
os.environ["PATH"] = "/usr/local/cuda-12.4/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12.4/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Verify GPU and nvcc
!nvidia-smi
!nvcc --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libopencv-dev is already the newest version (4.5.4+dfsg-9ubuntu4+jammy1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  pkgconf r-base-dev
The following NEW packages will be installed:
  pkg-config
0 upgraded, 1 newly installed, 2 to remove and 41 not upgraded.
Need to get 48.2 kB of archives.
After this operation, 10.2 kB disk space will be freed.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 pkg-config amd64 0.29.2-1ubuntu3 [48.2 kB]
Fetched 48.2 kB in 1s (71.5 kB/s)
(Reading databa

In [5]:
%%bash
# Write hello.cu
cat > hello.cu <<'EOF'
#include <cstdio>
#include <cuda_runtime.h>

// Simple 1D kernel
__global__ void hello_kernel() {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from thread %d in block %d (global %d)\n", threadIdx.x, blockIdx.x, idx);
}

int main() {
    int blocks = 2;
    int threadsPerBlock = 4;

    printf("Launching kernel with %d blocks x %d threads\n", blocks, threadsPerBlock);
    hello_kernel<<<blocks, threadsPerBlock>>>();
    cudaDeviceSynchronize();
    return 0;
}
EOF

# Compile & run
nvcc hello.cu -o hello
./hello


Launching kernel with 2 blocks x 4 threads
Hello from thread 0 in block 1 (global 4)
Hello from thread 1 in block 1 (global 5)
Hello from thread 2 in block 1 (global 6)
Hello from thread 3 in block 1 (global 7)
Hello from thread 0 in block 0 (global 0)
Hello from thread 1 in block 0 (global 1)
Hello from thread 2 in block 0 (global 2)
Hello from thread 3 in block 0 (global 3)


In [6]:
%%bash
# Write vector_add_compare.cu
cat > vector_add_compare.cu <<'EOF'
#include <iostream>
#include <chrono>
#include <cuda_runtime.h>

#define N 10000000 // 10 million elements

__global__ void vectorAdd(const float *A, const float *B, float *C, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) C[idx] = A[idx] + B[idx];
}

int main() {
    float *hA = new float[N];
    float *hB = new float[N];
    float *hC_cpu = new float[N];
    float *hC_gpu = new float[N];

    for(int i=0;i<N;i++) { hA[i]=i*0.001f; hB[i]=i*0.002f; }

    // CPU
    auto t0 = std::chrono::high_resolution_clock::now();
    for(int i=0;i<N;i++) hC_cpu[i] = hA[i]+hB[i];
    auto t1 = std::chrono::high_resolution_clock::now();
    double cpu_time = std::chrono::duration<double>(t1-t0).count();

    // Allocate device memory
    float *dA,*dB,*dC;
    cudaMalloc(&dA, N*sizeof(float));
    cudaMalloc(&dB, N*sizeof(float));
    cudaMalloc(&dC, N*sizeof(float));

    cudaMemcpy(dA, hA, N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, N*sizeof(float), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (N + threads - 1)/threads;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    vectorAdd<<<blocks, threads>>>(dA,dB,dC,N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_ms=0.0f;
    cudaEventElapsedTime(&gpu_ms,start,stop);
    cudaMemcpy(hC_gpu, dC, N*sizeof(float), cudaMemcpyDeviceToHost);

    // Verify
    bool correct = true;
    for(int i=0;i<10;i++){
        if(hC_cpu[i]!=hC_gpu[i]) { correct=false; break; }
    }

    std::cout << "CPU time: " << cpu_time << " s\n";
    std::cout << "GPU kernel time: " << gpu_ms/1000.0 << " s\n";
    std::cout << "Speedup: " << cpu_time/(gpu_ms/1000.0) << "x\n";
    std::cout << "Results match? " << (correct?"Yes":"No") << std::endl;

    // Cleanup
    delete[] hA; delete[] hB; delete[] hC_cpu; delete[] hC_gpu;
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    cudaEventDestroy(start); cudaEventDestroy(stop);

    return 0;
}
EOF

# Compile & run
nvcc vector_add_compare.cu -o vector_add_compare
./vector_add_compare


CPU time: 0.0641594 s
GPU kernel time: 0.0151918 s
Speedup: 4.2233x
Results match? Yes


In [7]:
# Upload an image first
from google.colab import files
uploaded = files.upload()  # Upload any JPG/PNG image (rename to test.jpg)


Saving 133911546634087743.jpg to 133911546634087743.jpg


In [13]:
from google.colab import files

# Upload the image you want to invert
uploaded = files.upload()  # Make sure to upload a .jpg or .png


Saving test.jpg.jpg to test.jpg (1).jpg


In [14]:
import os

# Rename the uploaded file to exactly 'test.jpg'
os.rename("test.jpg (1).jpg", "test.jpg")


In [15]:
import cv2

img = cv2.imread("test.jpg")
if img is None:
    print("Error: image not loaded!")
else:
    print("Image loaded successfully:", img.shape)


Image loaded successfully: (2848, 4272, 3)


In [16]:
%%bash
# Write image_invert.cu
cat > image_invert.cu <<'EOF'
#include <iostream>
#include <opencv2/opencv.hpp>
#include <cuda_runtime.h>
#include <chrono>
using namespace std;
using namespace cv;

__global__ void invertKernel(const unsigned char* in, unsigned char* out, int N){
    int idx = blockIdx.x*blockDim.x + threadIdx.x;
    if(idx<N) out[idx] = 255 - in[idx];
}

int main(int argc, char** argv){
    const char* fname = (argc>1)?argv[1]:"test.jpg";
    Mat img = imread(fname, IMREAD_COLOR);
    if(img.empty()){cerr<<"Cannot load image\n";return 1;}
    int H=img.rows, W=img.cols, C=img.channels();
    int size = H*W*C;
    cout<<"Image: "<<H<<"x"<<W<<" channels="<<C<<endl;

    Mat cpu_out = img.clone();
    Mat gpu_out = img.clone();

    // CPU inversion
    auto t0 = chrono::high_resolution_clock::now();
    for(int i=0;i<size;i++) cpu_out.data[i]=255-img.data[i];
    auto t1 = chrono::high_resolution_clock::now();
    double cpu_time = chrono::duration<double>(t1-t0).count();

    // GPU inversion
    unsigned char *d_in, *d_out;
    cudaMalloc(&d_in,size); cudaMalloc(&d_out,size);
    cudaMemcpy(d_in,img.data,size,cudaMemcpyHostToDevice);

    int threads=256;
    int blocks=(size+threads-1)/threads;
    cudaEvent_t start, stop; cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    invertKernel<<<blocks,threads>>>(d_in,d_out,size);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    float gpu_ms=0.0f; cudaEventElapsedTime(&gpu_ms,start,stop);
    cudaMemcpy(gpu_out.data,d_out,size,cudaMemcpyDeviceToHost);

    // Save
    imwrite("output_cpu.jpg", cpu_out);
    imwrite("output_gpu.jpg", gpu_out);

    // Verify
    bool match=true;
    for(int i=0;i<size;i++){if(cpu_out.data[i]!=gpu_out.data[i]){match=false;break;}}
    cout<<"CPU time: "<<cpu_time<<" s\nGPU kernel time: "<<gpu_ms/1000.0<<" s\nMatch? "<<(match?"Yes":"No")<<endl;

    cudaFree(d_in); cudaFree(d_out);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}
EOF

# Compile & run
nvcc image_invert.cu -o image_invert `pkg-config --cflags --libs opencv4`
./image_invert test.jpg


Image: 2848x4272 channels=3
CPU time: 0.0892142 s
GPU kernel time: 0.0140434 s
Match? Yes


/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::buildMaps" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/usr/include/opencv4/opencv2/stitching/detail/warpers.hpp(235): warning #611-D: overloaded virtual function "cv::detail::PlaneWarper::warp" is only partially overridden in class "cv::detail::AffineWarper"
  class AffineWarper : public PlaneWarper
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(100): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is only partially overridden in class "cv::detail::FeatherBlender"
  class FeatherBlender : public Blender
        ^

/usr/include/opencv4/opencv2/stitching/detail/blenders.hpp(127): warning #611-D: overloaded virtual function "cv::detail::Blender::prepare" is